# LFCC-CNN — Converted from your mfcc-cnn.ipynb

Every cell below is your original code. Lines that changed are marked with `# ✅ CHANGED` and lines that are new are marked with `# ✅ NEW`. Nothing else is touched.

## 1. Imports + Config

In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from scipy.fftpack import dct   # ✅ NEW — needed for DCT step in LFCC

In [2]:
ROOT_PATH = "/kaggle/input/datasets/artharking/torgod/TorgoD"  # change this

SR = 16000
N_LFCC = 20          # ✅ CHANGED: was N_MFCC = 20  — same number, just renamed
N_FILTER = 40        # ✅ NEW: number of linear triangular filters in the filterbank
FRAME_LEN = 0.025    # 25 ms — unchanged
HOP_LEN = 0.010      # 10 ms — unchanged

FIXED_LEN = 250   # unchanged

BATCH_SIZE = 32      # unchanged
EPOCHS = 20     # unchanged
LR = 1e-3            # unchanged

In [3]:
# unchanged
label_map = {
    "Low":  0,
    "Medium":1,
    "VeryLow":2
}
NUM_CLASSES = len(label_map)

## 2. Feature Extraction

### What changed and why

Your original `extract_mfcc()` called `librosa.feature.mfcc()` which internally:
1. Runs STFT
2. Applies a **Mel filterbank** (filters bunched at low frequencies)
3. Takes log
4. Applies DCT → gives 20 MFCC coefficients

The new `extract_lfcc()` does the **exact same 4 steps manually**, but in Step 2 uses a **Linear filterbank** (filters equally spaced across all frequencies). That single change is the entire difference between MFCC and LFCC.

**Output shape is identical: `(20, 250)` — so the CNN does not change at all.**

In [4]:
import os
import numpy as np
import librosa
from tqdm import tqdm

SAVE_PATH = "/kaggle/working/lfcc_features"   # ✅ CHANGED: was mfcc_features
os.makedirs(SAVE_PATH, exist_ok=True)


# ✅ NEW function — builds the linear triangular filterbank
# This replaces what librosa does internally for Mel filters
def build_linear_filterbank(n_filters, n_fft, sr):
    n_bins = n_fft // 2 + 1                        # number of FFT frequency bins
    freq_bins = np.linspace(0, sr / 2, n_bins)     # Hz value of each FFT bin

    # ← THIS is the key line: equally spaced filter edges in Hz
    # MFCC uses mel-spaced edges (bunched at low frequencies)
    # LFCC uses linearly-spaced edges (equal spacing everywhere)
    filter_points = np.linspace(0, sr / 2, n_filters + 2)

    fbank = np.zeros((n_filters, n_bins))

    for m in range(1, n_filters + 1):
        f_left   = filter_points[m - 1]   # left edge of triangle
        f_center = filter_points[m]       # peak of triangle
        f_right  = filter_points[m + 1]  # right edge of triangle

        for k, f in enumerate(freq_bins):
            if f_left <= f <= f_center:
                fbank[m - 1, k] = (f - f_left) / (f_center - f_left + 1e-10)
            elif f_center < f <= f_right:
                fbank[m - 1, k] = (f_right - f) / (f_right - f_center + 1e-10)

    return fbank   # shape: (40, n_bins)


# ✅ CHANGED: was extract_mfcc() — now extract_lfcc()
def extract_lfcc(file_path):
    y, sr = librosa.load(file_path, sr=SR)   # same as before

    n_fft      = int(FRAME_LEN * SR)         # 400 samples = 25ms window
    hop_length = int(HOP_LEN * SR)           # 160 samples = 10ms hop

    # Step 1: STFT → power spectrum  (same concept as inside librosa.feature.mfcc)
    stft       = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    power_spec = np.abs(stft) ** 2           # shape: (n_fft/2+1, T)

    # Step 2: Apply LINEAR filterbank  (← only difference from MFCC)
    fbank           = build_linear_filterbank(N_FILTER, n_fft, SR)
    filter_energies = np.dot(fbank, power_spec)   # shape: (40, T)

    # Step 3: Log compression  (same as inside librosa.feature.mfcc)
    log_energies = np.log(filter_energies + 1e-10)

    # Step 4: DCT → keep first N_LFCC coefficients  (same as inside librosa.feature.mfcc)
    lfcc = dct(log_energies, type=2, axis=0, norm='ortho')[:N_LFCC, :]
    # shape: (20, T) — identical to MFCC output shape

    # Pad or truncate to FIXED_LEN — exactly same as your original code
    if lfcc.shape[1] < FIXED_LEN:
        pad_width = FIXED_LEN - lfcc.shape[1]
        lfcc = np.pad(lfcc, ((0, 0), (0, pad_width)), mode='constant')
    else:
        lfcc = lfcc[:, :FIXED_LEN]

    return lfcc   # shape: (20, 250) — same as extract_mfcc() returned


# ✅ CHANGED: was save_mfcc_dataset — now save_lfcc_dataset
def save_lfcc_dataset(root_dir, split):
    split_path = os.path.join(root_dir, split)

    for severity in os.listdir(split_path):
        sev_path = os.path.join(split_path, severity)

        if not os.path.isdir(sev_path):
            continue

        save_sev_path = os.path.join(SAVE_PATH, split, severity)
        os.makedirs(save_sev_path, exist_ok=True)

        for file in tqdm(os.listdir(sev_path), desc=f"{split}-{severity}"):
            if not file.endswith(".wav"):
                continue

            file_path = os.path.join(sev_path, file)

            lfcc = extract_lfcc(file_path)   # ✅ CHANGED: was extract_mfcc

            save_file = os.path.join(save_sev_path, file.replace(".wav", ".npy"))
            np.save(save_file, lfcc)

In [5]:
save_lfcc_dataset(ROOT_PATH, "train")   # ✅ CHANGED: was save_mfcc_dataset
save_lfcc_dataset(ROOT_PATH, "test")    # ✅ CHANGED: was save_mfcc_dataset

test-Medium: 100%|██████████| 243/243 [00:04<00:00, 53.41it/s]


## 3. Dataset Class

In [6]:
import torch
from torch.utils.data import Dataset

# ✅ CHANGED: class name was MFCCDataset — now LFCCDataset
# Everything inside is identical — loads .npy files, returns (tensor, label)
class LFCCDataset(Dataset):
    def __init__(self, root_dir):
        self.files = []
        self.labels = []

        for severity in os.listdir(root_dir):
            sev_path = os.path.join(root_dir, severity)

            if not os.path.isdir(sev_path):
                continue

            for file in os.listdir(sev_path):
                if file.endswith(".npy"):
                    self.files.append(os.path.join(sev_path, file))
                    self.labels.append(label_map[severity])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        lfcc = np.load(self.files[idx])   # loads (20, 250) array

        lfcc  = torch.tensor(lfcc, dtype=torch.float32).unsqueeze(0)  # → (1, 20, 250)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return lfcc, label

In [7]:
# ✅ CHANGED: class name + folder path (mfcc_features → lfcc_features)
train_dataset = LFCCDataset("/kaggle/working/lfcc_features/train")
test_dataset  = LFCCDataset("/kaggle/working/lfcc_features/test")

In [8]:
# unchanged
val_size   = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size

train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

## 4. CNN Model — 100% Unchanged

The CNN receives shape `(batch, 1, 20, 250)`. LFCC also produces `(20, 250)` — same shape as MFCC — so the model needs zero changes.

In [9]:
# unchanged
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, 4, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 5, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 2),
            nn.ReLU(),
            nn.MaxPool2d((1, 2))
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 50))

        self.classifier = nn.Sequential(
            nn.Linear(128 * 1 * 50, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [10]:
# unchanged
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model     = CNNModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [11]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


## 5. Training Loop — unchanged except model save filename

In [12]:
import os
import shutil

best_val_acc = 0
save_path = "/tmp/best_model_lfcc.pth"   # ✅ CHANGED: filename reflects lfcc

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds   = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}: Loss={train_loss:.4f}, Val Acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path, _use_new_zipfile_serialization=False)
        print(f"✅ Best model updated! Val Acc: {val_acc:.4f}")

shutil.copy(save_path, "./best_model_lfcc.pth")   # ✅ CHANGED: filename
print("✅ Model copied to working directory!")

100%|██████████| 72/72 [00:05<00:00, 13.74it/s]


Epoch 1/20: Loss=53.9209, Val Acc=0.8504
✅ Best model updated! Val Acc: 0.8504


100%|██████████| 72/72 [00:03<00:00, 22.99it/s]


Epoch 2/20: Loss=27.2649, Val Acc=0.9016
✅ Best model updated! Val Acc: 0.9016


100%|██████████| 72/72 [00:03<00:00, 23.08it/s]


...

Epoch 18/20: Loss=1.1897, Val Acc=0.9724


100%|██████████| 72/72 [00:03<00:00, 22.23it/s]


Epoch 19/20: Loss=1.7643, Val Acc=0.9606


100%|██████████| 72/72 [00:03<00:00, 22.33it/s]

Epoch 20/20: Loss=1.5182, Val Acc=0.9685
✅ Model copied to working directory!


## 6. Predict Function

In [13]:
def predict(file_path, model_path="best_model_lfcc.pth"):   # ✅ CHANGED: filename
    model = CNNModel()
    model.load_state_dict(torch.load(model_path))
    model.eval()

    lfcc = extract_lfcc(file_path)                          # ✅ CHANGED: was extract_mfcc
    lfcc = torch.tensor(lfcc).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        output = model(lfcc)
        pred   = torch.argmax(output, dim=1).item()

    inv_map = {v: k for k, v in label_map.items()}
    return inv_map[pred]

## 7. Test Evaluation — unchanged

In [14]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [15]:
model = CNNModel().to(device)
model.load_state_dict(torch.load("best_model_lfcc.pth"))   # ✅ CHANGED: filename
model.eval()

CNNModel(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(4, 4), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 128, kernel_size=(2, 2), stride=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=(1, 2), stride=(1, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (adaptive_pool): AdaptiveAvgPool2d(output_size=(1, 50))
  (classifier): Sequential(
    (0): Linear(in_features=6400, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [16]:
# unchanged
correct    = 0
total      = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for x, y in tqdm(test_loader):
        x, y = x.to(device), y.to(device)

        outputs = model(x)
        preds   = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()
        total   += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

test_acc = correct / total
print(f"✅ Test Accuracy: {test_acc:.4f}")

100%|██████████| 20/20 [00:00<00:00, 96.23it/s]

✅ Test Accuracy: 0.9891


In [17]:
# unchanged
from sklearn.metrics import confusion_matrix, classification_report

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=label_map.keys()))


Classification Report:

              precision    recall  f1-score   support

         Low       0.96      1.00      0.98        53
      Medium       0.99      0.98      0.99       243
     VeryLow       0.99      0.99      0.99       344

    accuracy                           0.99       640
   macro avg       0.98      0.99      0.99       640
weighted avg       0.99      0.99      0.99       640



In [18]:
# import pandas as pd

# inv_map = {v: k for k, v in label_map.items()}

# filenames = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

# scores = []
# all_preds = []
# all_labels = []

# model.eval()
# with torch.no_grad():
#     for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
#         x, y = x.to(device), y.to(device)
#         outputs = model(x)
#         probs = torch.softmax(outputs, dim=1)
#         best_scores = probs.max(dim=1).values
#         preds = torch.argmax(outputs, dim=1)

#         scores.extend(best_scores.cpu().numpy())
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(y.cpu().numpy())

# df = pd.DataFrame({
#     "filename":      filenames,
#     "score":         scores,
#     "predict class": [inv_map[p] for p in all_preds],
#     "actual class":  [inv_map[l] for l in all_labels],
# })

# df.to_excel("/kaggle/working/results.xlsx", index=False)
# print("✅ Saved results.xlsx")

import pandas as pd

inv_map = {v: k for k, v in label_map.items()}

# Class order: Normal → High → Mid → Low → Very_Low
class_order = [ "Low","Medium", "VeryLow"]

filenames  = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

all_preds      = []
all_labels     = []
all_raw_scores = []   # raw logits per class
all_softmax    = []   # softmax probabilities per class

model.eval()
with torch.no_grad():
    for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
        x, y = x.to(device), y.to(device)
        outputs = model(x)                          # raw logits: (batch, NUM_CLASSES)
        probs   = torch.softmax(outputs, dim=1)     # softmax probabilities
        preds   = torch.argmax(outputs, dim=1)

        all_raw_scores.extend(outputs.cpu().numpy())
        all_softmax.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Build DataFrame
df = pd.DataFrame({"filename": filenames})

# Raw logit score for each class
for cls_name in class_order:
    df[f"score_{cls_name}"] = [row[label_map[cls_name]] for row in all_raw_scores]

# Softmax probability for each class
for cls_name in class_order:
    df[f"prob_{cls_name}"] = [row[label_map[cls_name]] for row in all_softmax]

# Final prediction & actual label
df["predict class"] = [inv_map[p] for p in all_preds]
df["actual class"]  = [inv_map[l] for l in all_labels]

# Save both CSV and Excel
df.to_csv("/kaggle/working/results.csv",   index=False)
df.to_excel("/kaggle/working/results.xlsx", index=False)
print("✅ Saved results.csv and results.xlsx")
print(df.head())


100%|██████████| 20/20 [00:00<00:00, 83.96it/s]


✅ Saved results.csv and results.xlsx
            filename  score_Low  score_Medium  score_VeryLow  prob_Low  \
0  M05-S1-L-0025.npy  29.847195     -5.059261     -22.559000   1.00000   
1  F01-S1-L-0012.npy   2.004627     -2.575048       0.531961   0.80673   
2  F01-S1-L-0022.npy   4.819469     -4.597988      -1.983681   0.99881   
3  F01-S1-L-0002.npy   0.425053     -0.065130       0.219716   0.41205   
4  M05-S1-L-0013.npy  24.913498     -3.513332     -19.323515   1.00000   

    prob_Medium  prob_VeryLow predict class actual class  
0  6.923390e-16  1.738914e-23           Low          Low  
1  8.275620e-03  1.849941e-01           Low          Low  
2  8.119569e-05  1.108950e-03           Low          Low  
3  2.523868e-01  3.355628e-01           Low          Low  
4  4.512160e-13  6.139155e-20           Low          Low  


## 8. Single File Prediction (unchanged logic)

In [19]:
# def predict(file_path, model, device):
#     model.eval()

#     lfcc = extract_lfcc(file_path)                                        # ✅ CHANGED
#     lfcc = torch.tensor(lfcc, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

#     with torch.no_grad():
#         output = model(lfcc)
#         pred = torch.argmax(output, dim=1).item()

#     inv_map = {v:k for k,v in label_map.items()}
#     return inv_map[pred]

# file = "/kaggle/input/YOUR_DATASET/test/Low/sample.flac"
# print("Prediction:", predict(file, model, device))